In [1]:
import mysql.connector

In [2]:
mydb=mysql.connector.connect(host="localhost",user="root",password="")
mycursor=mydb.cursor()

In [3]:
mycursor.execute("create database student_db")

In [4]:
mydb=mysql.connector.connect(host="localhost",user="root",password="",database="student_db")
mycursor=mydb.cursor()

In [9]:
mycursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(20),
    age INT,
    course VARCHAR(20)
)
""")

In [ ]:
import tkinter as tk
from tkinter import messagebox
cursor=mydb.cursor()

# Colors & Fonts
bg_color = "#f0f4f7"
btn_color = "#4CAF50"
btn_text = "white"
font_label = ("Arial", 11)
font_entry = ("Arial", 11)
font_btn = ("Arial", 10, "bold")

# Functions
def add_student():
    if entry_name.get() == "" or entry_age.get() == "" or entry_course.get() == "":
        messagebox.showwarning("Warning", "All fields are required")
        return

    cursor.execute(
        "INSERT INTO students (name, age, course) VALUES (%s, %s, %s)",
        (entry_name.get(), entry_age.get(), entry_course.get())
    )
    mydb.commit()

    messagebox.showinfo("Success", "Student Added")
    clear_fields()
    show_students()

def show_students():
    listbox.delete(0, tk.END)
    cursor.execute("SELECT * FROM students")
    for row in cursor.fetchall():
        listbox.insert(tk.END, row)

def delete_student():
    selected = listbox.curselection()
    if not selected:
        return

    student = listbox.get(selected)
    cursor.execute("DELETE FROM students WHERE id=%s", (student[0],))
    mydb.commit()

    messagebox.showinfo("Deleted", "Student Deleted")
    show_students()

def update_student():
    selected = listbox.curselection()
    if not selected:
        return

    student = listbox.get(selected)

    cursor.execute(
        "UPDATE students SET name=%s, age=%s, course=%s WHERE id=%s",
        (entry_name.get(), entry_age.get(), entry_course.get(), student[0])
    )
    mydb.commit()

    messagebox.showinfo("Updated", "Student Updated")
    show_students()

def select_student(event):
    selected = listbox.curselection()
    if not selected:
        return

    student = listbox.get(selected)

    entry_name.delete(0, tk.END)
    entry_age.delete(0, tk.END)
    entry_course.delete(0, tk.END)

    entry_name.insert(0, student[1])
    entry_age.insert(0, student[2])
    entry_course.insert(0, student[3])

def clear_fields():
    entry_name.delete(0, tk.END)
    entry_age.delete(0, tk.END)
    entry_course.delete(0, tk.END)

# GUI
root = tk.Tk()
root.title("Student CRUD App")
root.geometry("600x600")
root.configure(bg=bg_color)

# Title
tk.Label(root, text="Student Management System",
         font=("Arial", 16, "bold"),
         bg=bg_color, fg="#333").grid(row=0, column=0, columnspan=3, pady=10)

# Labels & Entries
tk.Label(root, text="Name", font=font_label, bg=bg_color).grid(row=1, column=0, padx=10, pady=5, sticky="w")
entry_name = tk.Entry(root, font=font_entry)
entry_name.grid(row=1, column=1, padx=10, pady=5)

tk.Label(root, text="Age", font=font_label, bg=bg_color).grid(row=2, column=0, padx=10, pady=5, sticky="w")
entry_age = tk.Entry(root, font=font_entry)
entry_age.grid(row=2, column=1, padx=10, pady=5)

tk.Label(root, text="Course", font=font_label, bg=bg_color).grid(row=3, column=0, padx=10, pady=5, sticky="w")
entry_course = tk.Entry(root, font=font_entry)
entry_course.grid(row=3, column=1, padx=10, pady=5)

# Buttons
tk.Button(root, text="Add", bg=btn_color, fg=btn_text, font=font_btn, width=12,
          command=add_student).grid(row=1, column=2, padx=10)

tk.Button(root, text="Update", bg="#2196F3", fg="white", font=font_btn, width=12,
          command=update_student).grid(row=2, column=2, padx=10)

tk.Button(root, text="Delete", bg="#f44336", fg="white", font=font_btn, width=12,
          command=delete_student).grid(row=3, column=2, padx=10)

tk.Button(root, text="Show All", bg="#555", fg="white", font=font_btn, width=12,
          command=show_students).grid(row=4, column=1, pady=10)

# Listbox
listbox = tk.Listbox(root, width=70, font=("Courier", 10), bg="white")
listbox.grid(row=5, column=0, columnspan=3, padx=10, pady=10)
listbox.bind("<<ListboxSelect>>", select_student)

# Run
root.mainloop()